# 2 . Opciones exoticas y dependientes de trayectoria

Sobre el mismo motor de autoria programatica (`when`/`cashflow`/`trigger`/`if_`/`both`,
`PLAN_PRODUCTS.md §7`), este notebook construye familias de productos que Black-Scholes no
puede precisar en cerrado: barreras (knock-in/knock-out, single y double), un digital/binario,
un asiatico aritmetico nativo (`qd.average`, `PLAN_IMPROVE_NOTEBOOK.md` Fase 2 -- con la replica
manual via `add`/`div` como celda de verificacion cruzada, no como unico camino), un lookback
real (`qd.running_max`/`qd.running_min`, imposible de escribir antes de esa misma fase), una
estrategia take-profit/stop-loss, y combinaciones (`straddle`/`strangle`) via `both`. Todo se
precia con `PayoffPriceQ` (y `PayoffHitProbabilityQ` para probabilidades de toque), y la
identidad *knock-in + knock-out = vanilla* sirve de test de regresion visual.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, "../../../build/clients/python")
sys.path.insert(0, "../src")

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import engine
import quantdesk as qd
from quantdesk import greeks

print("modulo engine importado desde:", engine.__file__)


modulo engine importado desde: S:\Projects\engine_quant\clients\python\notebooks\../../../build/clients/python\engine.cp312-win_amd64.pyd


## 1. Mercado y plantillas de barrera


In [2]:
import math

S0, R, Q_DIV, SIGMA, T = 180.0, 0.05, 0.006, 0.28, 1.0
OBS = "EQ.SPOT.AAPL"
MONITORING = [0.25, 0.5, 0.75, 1.0]

model = qd.Gbm(s0=S0, r=R, q=Q_DIV, sigma=SIGMA, observable=OBS)
market = qd.Market(pillars=[T], zero_rates=[R])
eng = qd.Engine(backend="cpu", n_paths=150_000, n_steps=1, seed=23)


def vanilla_call_contract(strike):
    """Delega en quantdesk.payoff.call_leg (PLAN_IMPROVE_NOTEBOOK2.md Fase 6): unica fuente
    de verdad para una pata call vainilla, compartida con 09_option_strategies_and_greeks.ipynb
    -- antes reimplementada aqui a mano (ver vanilla_call_contract_manual, celda de verificacion
    cruzada, no el unico camino)."""
    return qd.call_leg(OBS, strike, 1.0, T)


def vanilla_call_contract_manual(strike):
    """Version local horneada a mano (PRE-Fase 6, celda de verificacion cruzada): la que usaba
    este notebook antes de que quantdesk.payoff expusiera call_leg."""
    return qd.when(T, qd.cashflow("USD", qd.maximum(qd.fixing(OBS, T) - strike, 0.0)))


def up_and_in(event_id, barrier, underlying):
    condition = qd.greater_equal(qd.current(OBS), barrier)
    return qd.trigger(event_id, MONITORING, condition, "discrete", "at_hit", 0, True, underlying, qd.zero())


def up_and_out(event_id, barrier, underlying):
    condition = qd.greater_equal(qd.current(OBS), barrier)
    return qd.trigger(event_id, MONITORING, condition, "discrete", "at_hit", 0, True, qd.zero(), underlying)


def down_and_in(event_id, barrier, underlying):
    condition = qd.less_equal(qd.current(OBS), barrier)
    return qd.trigger(event_id, MONITORING, condition, "discrete", "at_hit", 0, True, underlying, qd.zero())


def double_knock_out(event_id, low, high, underlying):
    spot = qd.current(OBS)
    condition = qd.any_of([qd.less_equal(spot, low), qd.greater_equal(spot, high)])
    return qd.trigger(event_id, MONITORING, condition, "discrete", "at_hit", 0, True, qd.zero(), underlying)


def price_contract(contract, trade_id, measures=("PayoffPriceQ",)):
    trade = qd.PayoffProduct(id=trade_id, contract=contract)
    return eng.price(trade, model, market, list(measures))


# Verificacion cruzada: call_leg (motor) vs horneado a mano deben dar el MISMO precio exacto
# (misma seed/n_paths, PLAN_IMPROVE_NOTEBOOK2.md Fase 6).
call_native_check = price_contract(vanilla_call_contract(S0), "CALL_NATIVE_CHECK")["PayoffPriceQ"].scalar
call_manual_check = price_contract(vanilla_call_contract_manual(S0), "CALL_MANUAL_CHECK")["PayoffPriceQ"].scalar
assert call_native_check == call_manual_check, f"call_leg={call_native_check} manual={call_manual_check}"
print(f"call_leg vs horneado a mano: {call_native_check:.6f} == {call_manual_check:.6f} (coinciden exactamente)")

print(f"S0={S0}  sigma={SIGMA:.0%}  monitoring={MONITORING}")


call_leg vs horneado a mano: 23.500697 == 23.500697 (coinciden exactamente)
S0=180.0  sigma=28%  monitoring=[0.25, 0.5, 0.75, 1.0]


## 2. Knock-in + knock-out = vanilla

Exactamente una de las dos ramas paga en cada trayectoria: la suma de precios (evaluados con
la MISMA `PricingContext`/semilla, pero como dos simulaciones independientes, no numeros
aleatorios comunes explicitos) debe reproducir el precio vanilla dentro del ruido Monte Carlo.


In [3]:
barrier = S0 * 1.15
vanilla_contract = vanilla_call_contract(S0)
ui_contract = up_and_in("UI", barrier, vanilla_contract)
uo_contract = up_and_out("UO", barrier, vanilla_contract)

vanilla_price = price_contract(vanilla_contract, "VANILLA")["PayoffPriceQ"].scalar
ui_price = price_contract(ui_contract, "UI")["PayoffPriceQ"].scalar
uo_price = price_contract(uo_contract, "UO")["PayoffPriceQ"].scalar

labels = ["Vanilla", "Up-and-in", "Up-and-out", "UI + UO"]
values = [vanilla_price, ui_price, uo_price, ui_price + uo_price]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors, text=[f"{v:.2f}" for v in values], textposition="outside"))
fig.update_layout(
    title=f"Barrera up-and-in/out a {barrier:.0f} (115% del spot)",
    yaxis_title="precio",
    template="plotly_white",
)
fig.show()

print(f"vanilla={vanilla_price:.3f}  UI+UO={ui_price + uo_price:.3f}  diferencia={abs(vanilla_price - (ui_price + uo_price)):.3f}")


vanilla=23.501  UI+UO=23.616  diferencia=0.115


## 3. Precio y probabilidad de toque frente al nivel de barrera

Una sola llamada a `Engine.price` puede pedir varias medidas Monte Carlo a la vez sobre la
MISMA simulacion (`PayoffPriceQ` + `PayoffHitProbabilityQ`) -- no hace falta relanzar la
simulacion para cada medida.


In [4]:
barrier_levels = np.arange(1.02, 1.51, 0.04) * S0
prices, hit_probs = [], []
for b in barrier_levels:
    contract = up_and_in("UI", float(b), vanilla_contract)
    result = price_contract(contract, f"UI_{b:.0f}", measures=("PayoffPriceQ", ("PayoffHitProbabilityQ", {"event": "UI"})))
    prices.append(result["PayoffPriceQ"].scalar)
    hit_probs.append(result["PayoffHitProbabilityQ"].scalar)

# Dos magnitudes de escala distinta (precio en USD vs probabilidad en [0,1]): subplots apilados
# con eje X compartido, nunca doble eje Y (twinx) en el mismo panel.
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08)
fig.add_trace(go.Scatter(x=barrier_levels, y=prices, mode="lines+markers", name="precio up-and-in", line=dict(color="royalblue")), row=1, col=1)
fig.add_hline(
    y=vanilla_price,
    line=dict(color="royalblue", dash="dot"),
    annotation_text="precio vanilla (limite si barrier->spot)",
    row=1,
    col=1,
)
fig.add_trace(go.Scatter(x=barrier_levels, y=hit_probs, mode="lines+markers", name="P(toque)", line=dict(color="firebrick", dash="dash")), row=2, col=1)

fig.update_yaxes(title_text="precio up-and-in", row=1, col=1)
fig.update_yaxes(title_text="P(toque)", row=2, col=1)
fig.update_xaxes(title_text="nivel de barrera", row=2, col=1)
fig.update_layout(
    title="Up-and-in call: precio y probabilidad de toque vs nivel de barrera",
    template="plotly_white",
    height=650,
)
fig.show()


## 4. Corredor double knock-out: mas ancho, mas se acerca a la vanilla


In [5]:
half_widths = np.arange(0.06, 0.41, 0.03)
corridor_prices = []
for hw in half_widths:
    contract = double_knock_out("KO", S0 * (1.0 - hw), S0 * (1.0 + hw), vanilla_contract)
    corridor_prices.append(price_contract(contract, f"KO_{hw:.2f}")["PayoffPriceQ"].scalar)

fig = go.Figure()
fig.add_trace(go.Scatter(x=half_widths * 100.0, y=corridor_prices, mode="lines+markers", name="corredor double KO"))
fig.add_hline(y=vanilla_price, line=dict(color="grey", dash="dash"), annotation_text="vanilla (limite corredor infinito)")
fig.update_layout(
    title="Double knock-out call: precio vs ancho del corredor",
    xaxis_title="semi-ancho del corredor (% del spot)",
    yaxis_title="precio",
    template="plotly_white",
)
fig.show()


## 5. Digital/binario: `if_` + predicado

Un digital paga un importe fijo si el fixing final supera el strike -- construido con
`qd.if_(condicion, cashflow, zero())`, sin necesidad de ningun nodo nuevo.

In [6]:
PAYOUT = 10.0
strikes = np.arange(150.0, 221.0, 10.0)
digital_trades = []
for k in strikes:
    contract = qd.when(T, qd.if_(qd.greater(qd.fixing(OBS, T), float(k)), qd.cashflow("USD", PAYOUT), qd.zero()))
    digital_trades.append(qd.PayoffProduct(id=f"DIG_{k:.0f}", contract=contract))

rows = eng.price_many(digital_trades, model, market, ["PayoffPriceQ"])
digital_prices = [row.measures["PayoffPriceQ"].scalar for row in rows]

fig = go.Figure()
fig.add_trace(go.Scatter(x=strikes, y=digital_prices, mode="lines+markers", name="digital", line=dict(color="mediumpurple")))
fig.add_hline(y=PAYOUT * math.exp(-R * T), line=dict(color="grey", dash="dot"), annotation_text="cota superior: payout descontado")
fig.update_layout(
    title=f"Digital cash-or-nothing (payout={PAYOUT})",
    xaxis_title="strike",
    yaxis_title="precio del digital",
    template="plotly_white",
)
fig.show()


## 6. Asiatico aritmetico: nodo nativo `qd.average`

Desde `PLAN_IMPROVE_NOTEBOOK.md` Fase 2, `Average` esta cableado al compilador Monte Carlo
del payoff (`rust/crates/engine-core/src/payoff/compile.rs`): es una suma PONDERADA sobre un
`schedule` fijo (`sum(weights[i] * fixing(observable, schedule[i]))`, misma forma que el AST
de autoria ya existente en C++/`docs/schema/engine.payoff/v1.schema.json` -- NO una media con
divisor implicito), asi que un asiatico equiponderado usa `weights = [1/n]*n`.

La replica manual horneada a mano (`(fixing(t1)+...+fixing(tn))/n` via `add`/`div`) se
mantiene como celda de verificacion cruzada, no como unico camino (mismo patron que las
Fases 0/5 de `PLAN_IMPROVE_NOTEBOOK.md`): con la MISMA semilla/numero de trayectorias, ambos
caminos coinciden EXACTAMENTE (no solo dentro de ruido Monte Carlo), tal como exige el
criterio de aceptacion de la Fase 2.

In [7]:
def asian_call_contract(n_dates, strike):
    """qd.average nativo: suma ponderada equiponderada (weights=1/n) sobre un schedule fijo."""
    fixing_times = list(np.linspace(T / n_dates, T, n_dates))
    weights = [1.0 / n_dates] * n_dates
    average = qd.average(OBS, fixing_times, weights)
    return qd.when(T, qd.cashflow("USD", qd.maximum(average - strike, 0.0)))


def asian_call_contract_manual(n_dates, strike):
    """Replica horneada a mano (celda de verificacion cruzada, NO el unico camino): misma
    formula, construida con add/div sobre 'fixing' en vez del nodo 'average' nativo."""
    fixing_times = list(np.linspace(T / n_dates, T, n_dates))
    terms = [qd.fixing(OBS, t) for t in fixing_times]
    total = terms[0]
    for term in terms[1:]:
        total = total + term
    average = total / float(n_dates)
    return qd.when(T, qd.cashflow("USD", qd.maximum(average - strike, 0.0)))


# Verificacion cruzada: nativo vs horneado a mano deben coincidir EXACTAMENTE (misma seed/n_paths).
native_check = price_contract(asian_call_contract(4, S0), "ASIAN_NATIVE_CHECK")["PayoffPriceQ"].scalar
manual_check = price_contract(asian_call_contract_manual(4, S0), "ASIAN_MANUAL_CHECK")["PayoffPriceQ"].scalar
assert native_check == manual_check, f"nativo={native_check} manual={manual_check}"
print(f"average nativo vs horneado a mano (n=4): {native_check:.6f} == {manual_check:.6f} (coinciden exactamente)")

n_dates_grid = [1, 2, 4, 6, 12, 24]
asian_prices = []
for n in n_dates_grid:
    contract = asian_call_contract(n, S0)
    asian_prices.append(price_contract(contract, f"ASIAN_{n}")["PayoffPriceQ"].scalar)

fig = go.Figure()
fig.add_trace(go.Scatter(x=n_dates_grid, y=asian_prices, mode="lines+markers", name="asiatico", line=dict(color="saddlebrown")))
fig.add_hline(y=vanilla_price, line=dict(color="grey", dash="dash"), annotation_text="vanilla (n=1 equivalente)")
fig.update_layout(
    title="Asiatico aritmetico ATM (qd.average nativo): mas fechas -> menos volatilidad efectiva",
    xaxis_title="numero de fechas de promediado",
    yaxis_title="precio",
    template="plotly_white",
)
fig.show()


average nativo vs horneado a mano (n=4): 15.767402 == 15.767402 (coinciden exactamente)


## 7. Lookback real: `running_max`/`running_min` (`PLAN_IMPROVE_NOTEBOOK.md` Fase 2)

Antes de esta fase, un lookback (payoff que depende del maximo/minimo CORRIDO de la
trayectoria, no de un conjunto fijo de fechas) no tenia ningun workaround posible con los
nodos soportados. `qd.running_max`/`qd.running_min` ya compilan, con una decision de diseno
importante: replican el AST de autoria ya existente (`docs/schema/engine.payoff/v1.schema.json`,
`cpp/engine/include/engine/payoff/expression.hpp`) y solo llevan el `observable`, SIN un
`schedule` propio -- el motor Rust reduce en evaluacion sobre los instantes YA conocidos en
preflight (`CompiledPayoff::required_times()`) que sean `<=` el instante en el que se usa el
resultado (ver el doc-comment de `ScalarOp::RunningMin` en
`rust/crates/engine-core/src/payoff/ir.rs`).

Practicamente, eso significa que si ningun otro nodo del contrato referencia fechas
intermedias del observable, `required_times()` no tiene mas puntos que el propio vencimiento.
Para dar al lookback un schedule de monitorizacion real (aqui, semanal) se usa una
`qd.average` con `weights` a CERO como "ancla": no contribuye ningun importe (`0.0 * ancla`),
solo inyecta esas fechas en `required_times()` -- documentado, no oculto.

Un lookback call paga `max(running_max(S) - K, 0)`: como el maximo corrido siempre domina al
fixing final, su precio es >= el de la vanilla del mismo strike -- y crece con la frecuencia
de monitorizacion (mas oportunidades de capturar un pico mas alto). El lookback put
(`running_min`) es simetrico.

In [8]:
def _monitoring_anchor(dates):
    """'Ancla' de monitorizacion: un qd.average con weights a cero, que no contribuye ningun
    importe pero inyecta 'dates' en required_times() -- ver la nota de la seccion. Patron
    documentado en el doc-comment de ScalarOp::RunningMin (ir.rs)."""
    return qd.average(OBS, dates, [0.0] * len(dates))


def lookback_call_contract(monitoring_dates, strike):
    seeded_running_max = qd.running_max(OBS) + 0.0 * _monitoring_anchor(monitoring_dates)
    return qd.when(T, qd.cashflow("USD", qd.maximum(seeded_running_max - strike, 0.0)))


def lookback_call_contract_manual(monitoring_dates, strike):
    """Replica horneada a mano (celda de verificacion cruzada): encadena qd.maximum sobre los
    mismos fixings, sin usar el nodo 'running_max' nativo."""
    running_max = qd.fixing(OBS, monitoring_dates[0])
    for t in monitoring_dates[1:]:
        running_max = qd.maximum(running_max, qd.fixing(OBS, t))
    return qd.when(T, qd.cashflow("USD", qd.maximum(running_max - strike, 0.0)))


weekly_dates = list(np.linspace(T / 52, T, 52))

# Verificacion cruzada: nativo vs horneado a mano deben coincidir EXACTAMENTE.
lookback_native_check = price_contract(
    lookback_call_contract(weekly_dates, S0), "LOOKBACK_NATIVE_CHECK"
)["PayoffPriceQ"].scalar
lookback_manual_check = price_contract(
    lookback_call_contract_manual(weekly_dates, S0), "LOOKBACK_MANUAL_CHECK"
)["PayoffPriceQ"].scalar
assert lookback_native_check == lookback_manual_check, (
    f"nativo={lookback_native_check} manual={lookback_manual_check}"
)
print(
    f"running_max nativo vs horneado a mano (52 fechas): "
    f"{lookback_native_check:.6f} == {lookback_manual_check:.6f} (coinciden exactamente)"
)

monitoring_grid = [2, 4, 12, 26, 52]
lookback_prices = []
for n in monitoring_grid:
    dates = list(np.linspace(T / n, T, n))
    lookback_prices.append(price_contract(lookback_call_contract(dates, S0), f"LOOKBACK_{n}")["PayoffPriceQ"].scalar)

fig = go.Figure()
fig.add_trace(go.Scatter(x=monitoring_grid, y=lookback_prices, mode="lines+markers", name="lookback call (running_max)", line=dict(color="teal")))
fig.add_hline(y=vanilla_price, line=dict(color="grey", dash="dash"), annotation_text="vanilla (misma call ATM)")
fig.update_layout(
    title="Lookback call ATM: mas monitorizacion -> mayor maximo corrido esperado -> mas caro",
    xaxis_title="numero de fechas de monitorizacion",
    yaxis_title="precio",
    template="plotly_white",
)
fig.show()

assert all(p >= vanilla_price - 1e-6 for p in lookback_prices), "el lookback deberia dominar siempre a la vanilla"

running_max nativo vs horneado a mano (52 fechas): 41.946712 == 41.946712 (coinciden exactamente)


In [9]:
# running_min (lookback put): mismo patron, simetrico.
def lookback_put_contract(monitoring_dates, strike):
    seeded_running_min = qd.running_min(OBS) + 0.0 * _monitoring_anchor(monitoring_dates)
    return qd.when(T, qd.cashflow("USD", qd.maximum(strike - seeded_running_min, 0.0)))


lookback_put_price = price_contract(lookback_put_contract(weekly_dates, S0), "LOOKBACK_PUT")["PayoffPriceQ"].scalar
vanilla_put_price = price_contract(
    qd.when(T, qd.cashflow("USD", qd.maximum(S0 - qd.fixing(OBS, T), 0.0))), "VANILLA_PUT_CHECK"
)["PayoffPriceQ"].scalar
print(
    f"lookback put (running_min, 52 semanas)={lookback_put_price:.3f}  "
    f"vanilla put ATM={vanilla_put_price:.3f}"
)
assert lookback_put_price >= vanilla_put_price - 1e-6, "el lookback put deberia dominar siempre a la vanilla put"

lookback put (running_min, 52 semanas)=29.110  vanilla put ATM=15.964


## 8. Take-profit / stop-loss

Estrategia intradia/swing tipica: dos `trigger`s con prioridad (`priority`) y exclusion mutua
via `event_occurred` (patron de `docs/schema/engine.payoff/examples/tp_sl.json`). Se revisa
en `MONITORING` y paga la diferencia frente al precio de entrada en el primer evento que
dispare.


In [10]:
def take_profit_stop_loss_contract(entry_price, quantity, tp_return, sl_return, monitoring_times):
    ratio = qd.div(qd.current(OBS), entry_price) - 1.0
    tp_condition = qd.all_of([qd.greater_equal(ratio, tp_return), qd.negate(qd.event_occurred("STOP_LOSS"))])
    sl_condition = qd.all_of([qd.less_equal(ratio, sl_return), qd.negate(qd.event_occurred("TAKE_PROFIT"))])
    tp_payoff = qd.cashflow("USD", quantity * (qd.event_value("TAKE_PROFIT", OBS) - entry_price))
    sl_payoff = qd.cashflow("USD", quantity * (qd.event_value("STOP_LOSS", OBS) - entry_price))
    take_profit = qd.trigger("TAKE_PROFIT", monitoring_times, tp_condition, "discrete", "at_hit", 10, True, tp_payoff, qd.zero())
    stop_loss = qd.trigger("STOP_LOSS", monitoring_times, sl_condition, "discrete", "at_hit", 20, True, sl_payoff, qd.zero())
    return qd.both([take_profit, stop_loss])


entry_price, quantity = S0, 100.0
monthly = [t / 12.0 for t in range(1, 13)]
tp_return = 0.15
sl_returns = np.arange(-0.20, -0.02, 0.02)
tp_sl_prices = []
for sl_return in sl_returns:
    contract = take_profit_stop_loss_contract(entry_price, quantity, tp_return, float(sl_return), monthly)
    tp_sl_prices.append(price_contract(contract, f"TPSL_{sl_return:.2f}")["PayoffPriceQ"].scalar)

fig = go.Figure()
fig.add_trace(go.Scatter(x=sl_returns * 100.0, y=tp_sl_prices, mode="lines+markers", name="TP/SL", line=dict(color="crimson")))
fig.update_layout(
    title=f"Take-profit fijo en +{tp_return:.0%}: valor vs nivel de stop-loss",
    xaxis_title="nivel de stop-loss (%)",
    yaxis_title="valor esperado de la estrategia",
    template="plotly_white",
)
fig.show()


## 9. Combinaciones: straddle y strangle (`both`)

Un straddle (call+put mismo strike) y un strangle (call+put strikes distintos, fuera de
dinero) se construyen sin ningun concepto nuevo: `both([...])` suma dos contratos.


In [11]:
def vanilla_put_contract(strike):
    """Delega en quantdesk.payoff.put_leg (PLAN_IMPROVE_NOTEBOOK2.md Fase 6) -- ver
    vanilla_put_contract_manual para la version local horneada a mano (celda de verificacion
    cruzada, PRE-Fase 6)."""
    return qd.put_leg(OBS, strike, 1.0, T)


def vanilla_put_contract_manual(strike):
    return qd.when(T, qd.cashflow("USD", qd.maximum(strike - qd.fixing(OBS, T), 0.0)))


# Verificacion cruzada put: put_leg (motor) vs horneado a mano deben dar el MISMO precio exacto.
put_native_check = price_contract(vanilla_put_contract(S0), "PUT_NATIVE_CHECK")["PayoffPriceQ"].scalar
put_manual_check = price_contract(vanilla_put_contract_manual(S0), "PUT_MANUAL_CHECK")["PayoffPriceQ"].scalar
assert put_native_check == put_manual_check, f"put_leg={put_native_check} manual={put_manual_check}"
print(f"put_leg vs horneado a mano: {put_native_check:.6f} == {put_manual_check:.6f} (coinciden exactamente)")


def price_strategy(trade):
    """Precia un PayoffProduct (p.ej. el que devuelve qd.custom_strategy) sobre el mismo mercado
    que price_contract, sin duplicar el AST -- el combinador ya vive en custom_strategy."""
    return eng.price(trade, model, market, ["PayoffPriceQ"])


strangle_low, strangle_high = S0 * 0.9, S0 * 1.1
straddle_trade = qd.custom_strategy("STRADDLE", [qd.call_leg(OBS, S0, 1.0, T), qd.put_leg(OBS, S0, 1.0, T)])
strangle_trade = qd.custom_strategy(
    "STRANGLE", [qd.call_leg(OBS, strangle_high, 1.0, T), qd.put_leg(OBS, strangle_low, 1.0, T)]
)

straddle_price = price_strategy(straddle_trade)["PayoffPriceQ"].scalar
strangle_price = price_strategy(strangle_trade)["PayoffPriceQ"].scalar

# Verificacion cruzada de estrategia: custom_strategy(call_leg, put_leg) vs el patron previo de
# este notebook (vanilla_call_contract_manual/vanilla_put_contract_manual + qd.both([...]) a mano)
# deben dar el MISMO precio exacto.
straddle_manual_contract = qd.both([vanilla_call_contract_manual(S0), vanilla_put_contract_manual(S0)])
straddle_manual_price = price_contract(straddle_manual_contract, "STRADDLE_MANUAL_CHECK")["PayoffPriceQ"].scalar
assert straddle_price == straddle_manual_price, f"custom_strategy={straddle_price} manual={straddle_manual_price}"
print(
    f"custom_strategy(straddle) vs both([...]) a mano: "
    f"{straddle_price:.6f} == {straddle_manual_price:.6f} (coinciden exactamente)"
)

fig = go.Figure(
    go.Bar(
        x=["Straddle (ATM)", "Strangle (90/110)"],
        y=[straddle_price, strangle_price],
        marker_color=["#1f77b4", "#ff7f0e"],
        text=[f"{straddle_price:.2f}", f"{strangle_price:.2f}"],
        textposition="outside",
    )
)
fig.update_layout(
    title="Straddle vs strangle: apostar por volatilidad, mas barato lejos del dinero",
    yaxis_title="precio",
    template="plotly_white",
)
fig.show()


put_leg vs horneado a mano: 15.963952 == 15.963952 (coinciden exactamente)
custom_strategy(straddle) vs both([...]) a mano: 39.464649 == 39.464649 (coinciden exactamente)
